# MNIST Raw-LiRA Saturation Sweep v4 (holdout estimator)

**v4 changes vs v3 (2026-07-06 fixes, verified against deployed bytes -- see research/VALIDATION_2026-07-06.md sections 8-9):**
- **Sample-split holdout is the PRIMARY conservative bound** (fix #1): threshold selected on a disjoint selection half, Wilson bound certified on an untouched estimation half -> valid 95% coverage. The old in-sample bound is kept only as diagnostic columns (`*_insample`) and must never be quoted.
- **OUT-count matching in Raw-LiRA scoring** (fix #2): non-member references are subsampled to the member OUT-count distribution, removing the small-K variance asymmetry that inflated point estimates.
- Censoring flag renamed to `conservative_zero_below_floor_or_no_signal` (fix #3): a conservative 0 no longer presumes signal exists.
- Plateau verdict returns `NO VERDICT` when fewer than 2 non-censored valid cells exist (fix #3) instead of a default STILL-CLIMBING string.
- `compute_epsilon_pld` hard-defaults to the google `dp_accounting` backend and RAISES if it is missing (fix #5) -- the GDP-CLT fallback can never be silently mislabeled as PLD again.
- The bundle now ships `tests/test_empirical.py`; pre-flight RUNS the full suite (13 tests incl. 200-rep null coverage) and aborts on any failure.

Upload `saturation_bundle_v4.zip` when prompted.

Bundle sha256: `754fe0e98433b95bd74ddee20b1674396c5a4220b516eb5d4210861d1746a564`

In [ ]:
BUNDLE_B64 = (
    "..."
    "BPMDAABQSwUGAAAAADwAPACKGQAATxEBAAAA"
)

# --- Self-contained bundle staging (no files.upload widget) --------------------
# The bundle is embedded above as base64 and verified against the same sha256 the
# original notebook pinned. Identical bytes, identical integrity check.
import base64, hashlib, io, zipfile
data = base64.b64decode(BUNDLE_B64)
digest = hashlib.sha256(data).hexdigest()
expected = "754fe0e98433b95bd74ddee20b1674396c5a4220b516eb5d4210861d1746a564"
assert digest == expected, f"BUNDLE MISMATCH: {digest} != {expected}"
with open("saturation_bundle_v4.zip", "wb") as fh:
    fh.write(data)
zipfile.ZipFile(io.BytesIO(data)).extractall(".")
# Colab now runs Python 3.12, whose unittest dropped implicit-namespace-package
# support: `python -m unittest tests.test_empirical` cannot import tests/ without
# an __init__.py. The bundle predates 3.12. Make tests/ a real package so the
# 13-test pre-flight suite (unchanged) can run.
import pathlib
pathlib.Path("tests/__init__.py").touch()
print("sha256 verified:", digest)
import os
print(sorted(os.listdir(".")))


In [ ]:
%pip -q install opacus dp-accounting pyyaml scipy
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY -- ABORT: runtime will be ~10-20x slower; Runtime > Change runtime type > T4 GPU")

In [ ]:
# --- PRE-FLIGHT 1/2: run the bundled test suite (aborts on any failure) ------
# 13 tests incl. HoldoutThresholdSelectionTests null coverage (200 pure-noise
# reps -> holdout conservative eps must be 0 in >=95%).
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "unittest", "tests.test_empirical", "-v"],
                   capture_output=True, text=True)
print(r.stderr[-3000:])
assert r.returncode == 0, "TEST SUITE FAILED -- do NOT run the experiment."
print("Bundled test suite: 13/13 PASS")

In [ ]:
# --- PRE-FLIGHT 2/2: v4 fix markers + estimator/scorer sanity (aborts) ------
import sys, random, statistics, inspect
sys.path.insert(0, "src"); sys.path.insert(0, "codex")
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.privacy.pld_accounting import compute_epsilon_pld
import run_raw_lira_pilot as pilot

checks = []
# (a) holdout path exists, is wired as PRIMARY in build_result_row, and works
sig = inspect.signature(estimate_empirical_lower_bound)
checks.append(("estimator exposes threshold_selection", "threshold_selection" in sig.parameters))
brr = inspect.getsource(pilot.build_result_row)
checks.append(("build_result_row primary = holdout", 'threshold_selection="holdout"' in brr))
checks.append(("build_result_row keeps in-sample as diagnostic", "epsilon_lower_conservative_insample" in brr))
random.seed(608)
kw = dict(delta=1e-5, align_event_to_score_direction=True, require_member_favoring=True,
          report_confidence_supported_lower_bound=True, score_direction="higher")
m = [random.gauss(0, 1) for _ in range(640)]; n = [random.gauss(0, 1) for _ in range(640)]
e0 = estimate_empirical_lower_bound(member_scores=m, nonmember_scores=n,
                                    threshold_selection="holdout", **kw)
checks.append(("holdout null -> eps=0", e0.epsilon_lower_empirical_conservative == 0.0))
checks.append(("holdout method tag", e0.threshold_selection == "holdout"))
m = [random.gauss(3, 1) for _ in range(640)]; n = [random.gauss(0, 1) for _ in range(640)]
e1 = estimate_empirical_lower_bound(member_scores=m, nonmember_scores=n,
                                    threshold_selection="holdout", **kw)
checks.append(("holdout signal -> eps>0", e1.epsilon_lower_empirical_conservative > 0.5))
# (b) OUT-count matching active in the scorer
rls_sig = inspect.signature(pilot.raw_lira_scores)
checks.append(("scorer has match_out_counts=True default",
               rls_sig.parameters["match_out_counts"].default is True))
src = inspect.getsource(pilot.raw_lira_scores)
checks.append(("bundle has SYMMETRIC scorer", "fmean(out_losses) - float(target_" in src.replace("\n", "")))
# (c) disjoint sampler + quality flags + NO-VERDICT fix
checks.append(("bundle has disjoint sampler", hasattr(pilot, "sample_query_indices_disjoint")))
checks.append(("bundle has quality flags", hasattr(pilot, "apply_quality_flags")))
checks.append(("censoring flag renamed", "conservative_zero_below_floor_or_no_signal" in inspect.getsource(pilot.apply_quality_flags)))
import run_mnist_saturation as sat
checks.append(("NO VERDICT guard present", "NO VERDICT" in inspect.getsource(sat._saturation_verdict)))
# (d) hard PLD backend: default google AND dp_accounting importable + true-PLD value
checks.append(("compute_epsilon_pld default backend=google",
               inspect.signature(compute_epsilon_pld).parameters["backend"].default == "google"))
res = compute_epsilon_pld(noise_multiplier=1.1, sampling_rate=128/2048, num_steps=16, delta=1e-5)
checks.append(("PLD backend is google_dp_accounting", res["backend_used"] == "google_dp_accounting"))

failed = [name for name, ok in checks if not ok]
for name, ok in checks: print(("PASS " if ok else "FAIL ") + name)
if failed:
    raise SystemExit(f"SANITY CHECKS FAILED: {failed} -- do NOT run the experiment.")
print("\nAll v4 pre-flight checks passed -- safe to run.")

In [ ]:
!python codex/run_mnist_saturation.py

In [ ]:
import json, pandas as pd
rows = json.load(open("codex/results/mnist_saturation/mnist_saturation_summary.json"))
df = pd.DataFrame([r for r in rows if r.get("attack") == "passive_raw_lira"])
cols = ["k_shadows", "epsilon_lower_conservative", "epsilon_lower_point",
        "epsilon_lower_conservative_insample", "epsilon_lower_gdp",
        "epsilon_upper_tighter", "tightness_ratio_tighter", "threshold_selection",
        "validity", "censored"]
display(df[[c for c in cols if c in df.columns]].sort_values("k_shadows"))
verdict = json.load(open("codex/results/mnist_saturation/mnist_saturation_verdict.json"))
print("VERDICT:", verdict["verdict"])
print("last delta:", verdict["last_delta_vs_prev_k"], "| ladder cells:", verdict.get("num_ladder_cells"))

In [ ]:
import matplotlib.pyplot as plt
ok = df[(df.validity == "ok") & (df.censored == "not_censored")]
if len(ok) == 0:
    print("All cells censored (conservative 0 at every K) -- nothing to plot; report as below detection floor.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].semilogx(ok.k_shadows, ok.epsilon_lower_conservative, "o-", base=2, label="holdout Wilson conservative")
    ax[0].semilogx(ok.k_shadows, ok.epsilon_lower_point, "s--", base=2, label="holdout point (diagnostic)")
    ax[0].axhline(ok.epsilon_upper_tighter.iloc[0], color="r", ls=":", label="eps_upper (PLD)")
    ax[0].set_xlabel("K shadows"); ax[0].set_ylabel("eps_lower"); ax[0].legend()
    ax[0].set_title("Saturation ladder (valid, non-censored)")
    ax[1].semilogx(ok.k_shadows, ok.tightness_ratio_tighter, "o-", base=2)
    ax[1].set_xlabel("K shadows"); ax[1].set_ylabel("tightness vs PLD"); ax[1].set_title("Tightness")
    plt.tight_layout(); plt.show()

In [ ]:
import json, shutil
verdict = json.load(open("codex/results/mnist_saturation/mnist_saturation_verdict.json"))
print("VERDICT:", verdict["verdict"])
for step in verdict["ladder"]:
    print("K=%s eps_lower=%s delta=%s tightness=%s" % (
        step["k_shadows"], step["epsilon_lower_conservative"],
        step["delta_vs_prev_k"], step["tightness_ratio_tighter"]))
shutil.make_archive("mnist_saturation_results_v4", "zip", "codex/results", "mnist_saturation")
from google.colab import files
files.download("mnist_saturation_results_v4.zip")
